# DHL Stakeholder QA — Agentic RAG System

Built over: *DHL Group Stakeholder Engagement Guideline*



## Install all dependencies

In [ ]:
# Install all required packages

!pip install -q langchain langchain-community langchain-groq chromadb \
               sentence-transformers pypdf ragas datasets \
               langsmith tavily-python python-dotenv pyyaml

print("All packages installed successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY      = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
TAVILY_API_KEY    = os.getenv("TAVILY_API_KEY")

# Validate all keys are present before proceeding
missing = [k for k, v in {
    "GROQ_API_KEY":      GROQ_API_KEY,
    "LANGCHAIN_API_KEY": LANGCHAIN_API_KEY,
    "TAVILY_API_KEY":    TAVILY_API_KEY
}.items() if not v]

if missing:
    raise EnvironmentError(
        f"Missing environment variables: {missing}\n"
        "Create a .env file from .env.example and upload it to Colab."
    )

os.environ["GROQ_API_KEY"]         = GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"]    = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "DHL-Stakeholder-RAG"
os.environ["TAVILY_API_KEY"]       = TAVILY_API_KEY

print("✅ All API keys loaded from .env")
print("🔍 LangSmith tracing enabled — project: DHL-Stakeholder-RAG")


✅ All API keys loaded from .env
🔍 LangSmith tracing enabled — project: DHL-Stakeholder-RAG


## Upload & parse the DHL PDF

Upload `stakeholder-engagement-guideline.pdf` using the file upload button that appears below, or mount from Google Drive.

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
import tempfile, shutil

# Upload the PDF
print("Upload the DHL Stakeholder Engagement Guideline PDF")
uploaded = files.upload()

# Save to a temp path and load
pdf_filename = list(uploaded.keys())[0]
with open(pdf_filename, 'wb') as f:
    f.write(uploaded[pdf_filename])

loader = PyPDFLoader(pdf_filename)
raw_docs = loader.load()

print(f"\n PDF loaded: {len(raw_docs)} pages")
for i, doc in enumerate(raw_docs):
    print(f"  Page {i+1}: {len(doc.page_content)} characters")

Upload the DHL Stakeholder Engagement Guideline PDF


Saving stakeholder-engagement-guideline.pdf to stakeholder-engagement-guideline.pdf

 PDF loaded: 3 pages
  Page 1: 1761 characters
  Page 2: 1992 characters
  Page 3: 778 characters


## Chunk & embed into ChromaDB

This is the **ingestion pipeline**: split → embed → store in vector DB.

In [ ]:
!pip install -q langchain-text-splitters

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

#  Step 1: Chunk the document
# chunk_size=500
# chunk_overlap=100 to preserve context
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(raw_docs)
print(f" Document split into {len(chunks)} chunks")

#  Step 2: Load embedding model
# BGE-small-en
print("\n Loading BGE embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("BGE-small-en-v1.5 loaded")

# Step 3: Store in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="dhl_stakeholder_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"\n ChromaDB vector store created with {len(chunks)} vectors")
print("\n Quick retrieval test:")
test_results = retriever.invoke("What are DHL's guiding principles for stakeholder engagement?")
for i, doc in enumerate(test_results):
    print(f"  Chunk {i+1} (page {doc.metadata.get('page',0)+1}): {doc.page_content[:120]}...")

 Document split into 7 chunks

 Loading BGE embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BGE-small-en-v1.5 loaded

 ChromaDB vector store created with 7 vectors

 Quick retrieval test:
  Chunk 1 (page 2): 4. Guiding principles for stakeholder engagement at DHL Group 
At DHL Group, our stakeholder engagement commitment is gu...
  Chunk 2 (page 2): 4. Guiding principles for stakeholder engagement at DHL Group 
At DHL Group, our stakeholder engagement commitment is gu...
  Chunk 3 (page 2): 4. Guiding principles for stakeholder engagement at DHL Group 
At DHL Group, our stakeholder engagement commitment is gu...


**System 1 — general Q&A over DHL document**

this system is a general Q&A system that answers general question about DHL Group's stakeholder engagement policies.
Two proposed RAGs are being compared in this system:

1.   The first relies on a template that has no structural guidance.
2.   The second relies on excplict instuctions.

**System 2 — triage agent**

this system generates reports in case an issue is raise by a stakeholder. The report include several information about the issue as well as an action plan to solve the problem raised by the stakeholder.


```
**Triage agent**'s pipline is as follows :

1)CLASSIFIER AGENT : Classify the type of the issue and route it to the department
2)RISK ASSESSOR AGENT :  Assess the level of urgency
3)RECOMMENDED COURSE OF ACTION AGENT : Enagage with stakeholder through :
  information and communication OR
  consultation OR
  formal stakeholder dialogues OR
  long-term partnerships

Additionally, a conditional edge is proposed after agent 2 :

*   if the the issue is critical then Action Advisor LLM is invoked.
*   else a direct report with standard response templates is generated.

```

The third agent recommend course of action. two proposed agents are being compared at this level:

1.   Agent 3.1 : Suggest a course of action without giving explicit instructions.
2.   Agent 3.2 : Suggest a course of action following pre-defined reasoning steps.




## make two prompts versions (v1 and v2)



In [ ]:
os.makedirs("prompts", exist_ok=True)

✅ prompts/ folder ready


In [53]:
import json
from datetime import datetime
import yaml
from pathlib import Path


# ── Prompt Registry ───────────────────────────────────────────────────────────
# Proper prompt versioning: every iteration is tracked with
#   - version number
#   - created date
#   - author
#   - change_log         : WHY it was changed (the most important field)
#   - status             : draft | active | deprecated
#   - eval_scores        : RAGAS scores (filled after evaluation step)
#   - template           : the actual prompt
#

# ── YAML prompt loader ────────────────────────────────────────
# In production: prompts/ folder is committed to Git.
# Every change to a prompt = a new YAML file + a Git tag.
# e.g. git tag prompt-action-advisor-v2.0
# This makes every prompt version reproducible and auditable.

PROMPTS_DIR = Path("prompts")  # folder containing all YAML prompt files

def load_prompt(name: str) -> dict:
    """
    Load a versioned prompt from its YAML file.

    Args:
        name: prompt name matching the YAML filename without extension
              e.g. 'rag_v1.0' loads prompts/rag_v1.0.yaml

    Returns:
        dict with keys: version, status, change_log, template, eval_scores, ...

    Raises:
        FileNotFoundError: if the YAML file does not exist
    """
    path = PROMPTS_DIR / f"{name}.yaml"
    if not path.exists():
        available = [f.stem for f in PROMPTS_DIR.glob("*.yaml")]
        raise FileNotFoundError(
            f"Prompt file '{path}' not found.\n"
            f"Available prompts: {available}"
        )
    with open(path, "r") as f:
        return yaml.safe_load(f)

def get_active_prompt(system: str) -> dict:
    """
    Get the currently active prompt for a given system prefix.
    Useful to always load the active version without hardcoding a name.

    Args:
        system: prefix to filter by e.g. 'rag' or 'action_advisor'

    Returns:
        dict of the active prompt with the highest version number
    """
    candidates = []
    for path in PROMPTS_DIR.glob(f"{system}_v*.yaml"):
        p = yaml.safe_load(open(path))
        if p.get("status") == "active":
            candidates.append((path.stem, p))
    if not candidates:
        raise ValueError(f"No active prompt found for system '{system}'")
    # Sort by version string descending — picks highest version if multiple active
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

def update_eval_scores(prompt_name: str, scores: dict):
    """
    Link RAGAS evaluation scores back to the prompt version that produced them.
    This is what closes the loop between versioning and evaluation.
    Call this after every RAGAS run.
    Example: update_eval_scores('rag_v2.0', {'faithfulness': 0.83, 'answer_relevancy': 0.87})
    """
    if prompt_name not in PROMPT_REGISTRY:
        raise ValueError(f"Prompt '{prompt_name}' not found.")
    PROMPT_REGISTRY[prompt_name]["eval_scores"] = {
        **scores,
        "evaluated_at": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    print(f"Scores linked to {prompt_name}: {scores}")

def print_registry_summary():
    """Print a full version history table across all prompts."""
    print("\n" + "="*75)
    print("  PROMPT VERSION REGISTRY")
    print("="*75)
    print(f"  {'Name':<25} {'Ver':<6} {'Status':<12} {'Scores':<20} {'Change log'}")
    print("-"*75)
    for name, p in PROMPT_REGISTRY.items():
        scores_str = (
            ", ".join(f"{k}: {v:.2f}" for k, v in p["eval_scores"].items()
                      if k != "evaluated_at")
            if p["eval_scores"] else "not evaluated yet"
        )
        status_emoji = {"active": "🟢", "deprecated": "🔴", "draft": "🟡"}.get(p["status"], "⚪")
        change_summary = p["change_log"][:45] + "..." if len(p["change_log"]) > 45 else p["change_log"]
        print(f"  {name:<25} {p['version']:<6} {status_emoji} {p['status']:<10} {scores_str:<20} {change_summary}")
    print("="*75)

def print_version_history(prefix: str):
    """
    Print the full history of a specific prompt family.
    Shows the evolution and reasoning behind each change.
    """
    versions = {k: v for k, v in PROMPT_REGISTRY.items() if k.startswith(prefix)}
    if not versions:
        print(f"No prompts found with prefix '{prefix}'")
        return
    print(f"\n Version history for: {prefix}")
    print("─"*65)
    for name, p in versions.items():
        status_emoji = {"active": "🟢", "deprecated": "🔴", "draft": "🟡"}.get(p["status"], "⚪")
        print(f"\n  {status_emoji} {name}  (created: {p['created']} by {p['author']})")
        print(f"  Why changed : {p['change_log']}")
        print(f"  Eval scores : {p['eval_scores'] if p['eval_scores'] else 'not yet evaluated'}")


# ── Display on load ───────────────────────────────────────────────────────────
print_registry_summary()
print()
print_version_history("rag")
print()
print_version_history("action_advisor")



  PROMPT VERSION REGISTRY
  Name                      Ver    Status       Scores               Change log
---------------------------------------------------------------------------
  rag_v1.0                  1.0    🟢 active     answer_relevancy: 0.96, faithfulness: 0.76, context_precision: 1.00 Initial version. Direct Q&A style with no str...
  rag_v2.0                  2.0    🟢 active     answer_relevancy: 0.94, faithfulness: 0.84, context_precision: 0.80 Introduced structured output instructions and...
  action_advisor_v1.0       1.0    🟢 active     guideline_faithfulness: 5.00, urgency_appropriateness: 5.00, action_concreteness: 5.00, overall_score: 5.00 Initial version. Direct instruction — tells t...
  action_advisor_v2.0       2.0    🟢 active     guideline_faithfulness: 4.00, urgency_appropriateness: 5.00, action_concreteness: 5.00, overall_score: 4.70 Added chain-of-thought reasoning steps that e...


 Version history for: rag
─────────────────────────────────────────────────

##  RAG chain (retrieval + generation)

Building the full pipeline.

Trying both prompt versions of System 1  and compare the difference in output quality.

In [54]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


#  LLM setup (Groq — free, fast)
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

def build_rag_chain(prompt_version: str):
    """Build a RAG chain with a specific prompt version."""
    prompt_config = load_prompt(prompt_version)
    prompt = PromptTemplate(
        template=prompt_config["template"],
        input_variables=["context", "question"]
    )
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain

#  Test both versions on the same question
test_question = "What are the guiding principles for stakeholder engagement at DHL?"

chain_v1 = build_rag_chain("rag_v1.0")
chain_v2 = build_rag_chain("rag_v2.0")

print(f" Question: {test_question}")
print("\n" + "─"*60)
print(" PROMPT v1 response:")
answer_v1 = chain_v1.invoke(test_question)
print(answer_v1)

print("\n" + "─"*60)
print(" PROMPT v2 response:")
answer_v2 = chain_v2.invoke(test_question)
print(answer_v2)

print("\n Both RAG chains working. Check smith.langchain.com to see the traces")

 Question: What are the guiding principles for stakeholder engagement at DHL?

────────────────────────────────────────────────────────────
 PROMPT v1 response:
The guiding principles for stakeholder engagement at DHL Group are:

1. Inclusivity: We strive to include in our engagement activities all stakeholders that are affected by or affect our operations.
2. Materiality: We are committed to identifying and prioritizing the relevant issues for stakeholders that have an impact on our operations.
3. Responsiveness: We aim to incorporate relevant outcomes and insights from engagement into our strategic decision-making and to respond to stakeholders appropriately.

────────────────────────────────────────────────────────────
 PROMPT v2 response:
According to Section 4 of the DHL Stakeholder Engagement Guideline, the guiding principles for stakeholder engagement at DHL Group are Inclusivity, Materiality, and Responsiveness. These principles are guided by the AA1000 SES Stakeholder Engageme

##  Agentic routing (retrieval vs web search)

in System 1 : ReAct agent reasons about which tool to use

*   Tool 1: Retrieve from DHL PDF  to answer the question.
*   Tool 2: Web search for questions outside the document.



In [56]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub

#  Tool 1: Retrieve from DHL PDF
def retrieve_from_dhl_docs(query: str) -> str:
    """Search the DHL Stakeholder Engagement Guideline for relevant information."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found in the DHL document."
    result = ""
    for i, doc in enumerate(docs):
        result += f"[Chunk {i+1}]\n{doc.page_content}\n\n"
    return result

dhl_retrieval_tool = Tool(
    name="dhl_document_search",
    func=retrieve_from_dhl_docs,
    description="Use this tool to answer questions about DHL Group's stakeholder engagement policies, principles, guidelines, or processes. Always try this tool first."
)

#  Tool 2: Web search for questions outside the document
web_search_tool = TavilySearchResults(
    max_results=2,
    description="Use this tool ONLY when the question is about general DHL news, current events, or topics not covered in the internal document."
)

tools = [dhl_retrieval_tool, web_search_tool]

#  Build the ReAct agent
# ReAct = Reasoning + Acting: the agent reasons about which tool to use
react_prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,       #  True to show the agent's reasoning steps
    max_iterations=4,
    handle_parsing_errors=True
)

#  Test 1: Question INSIDE the document ( hence it should be using dhl_document_search)
print("="*60)
print("TEST 1 — Question in the DHL document:")
print("="*60)
result1 = agent_executor.invoke({"input": "What is the AA1000SES process and how many stages does it have?"})
print("\n Final answer:", result1["output"])

print("\n" + "="*60)
print("TEST 2 — Question OUTSIDE the document (should be using web search):")
print("="*60)
result2 = agent_executor.invoke({"input": "Who is the current CEO of DHL Group?"})
print("\n Final answer:", result2["output"])

TEST 1 — Question in the DHL document:

 Final answer: The AA1000SES process has four stages: Planning the stakeholder engagement, Preparing the stakeholder engagement, Implementing the engagement format, and Reviewing and evaluating the engagement.

TEST 2 — Question OUTSIDE the document (should be using web search):

 Final answer: The current CEO of DHL Group is Tobias Meyer.


**System two** manages three specialized AI agents, to solve problems raised by the stakeholder.


Stakeholder Triage System pipeline :

1)CLASSIFIER AGENT : Classify the type of the issue and route it to the department.

2)RISK ASSESSOR AGENT :  Assess the level of urgency (Conditional edge)

    
   * if the the issue is critical then Action Advisor LLM is invoked.
   * else a direct report with standard response templates is generated.

3)RECOMMEND A COURSE OF ACTION AGENT (with two prompt versions)




                                 





**LangSmith tracing** : I added **custom trace metadata** to make the traces more informative. Eg :

 For each incoming stakeholder message you'll see the full execution tree — which conditional branch was taken, how long each agent spent, what each agent received as input from the shared state, and what it wrote back.

In [ ]:
!pip install -q langgraph
!pip install -qU langchain-groq

In [58]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, List
import operator, json, re

from langsmith import traceable

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

# 1. SHARED STATE
class TriageState(TypedDict):
    # Input
    stakeholder_message: str
    action_advisor_prompt_version: str

    # Filled by Agent 1 — Classifier
    issue_type: str           # Environmental | Compliance | Reputational | Strategic | Operational
    stakeholder_group: str    # NGO | Regulator | Investor | Media | Supplier .
    owning_department: str    # e.g. Corporate Sustainability Office

    # Filled by Agent 2 — Risk Assessor
    urgency_level: str        # Low | Medium | High | Critical
    risk_type: str            # PR | Regulatory | Strategic | Operational | Legal
    ramifications: str

    # Filled by Agent 3a — Action Advisor (if High/Critical path)
    # OR Agent 3b — Playbook Matcher (else Low/Medium path)
    engagement_format: str
    recommended_steps: str
    escalate_to_council: bool
    timeline: str
    path_taken: str           # 'llm_reasoning' or 'playbook'

    # Final report
    triage_report: str
    steps_taken: Annotated[List[str], operator.add]


# 2. PLAYBOOK
# Deterministic templates for Low/Medium urgency cases.
# Keyed by (stakeholder_group, issue_type).
# Fallback key '*' handles any unmatched combination.
# This is intentionally NOT an LLM call — to be fast, consistent, cheap.

PLAYBOOK = {
    ("NGO", "Environmental"): {
        "engagement_format": "Information & Communication",
        "recommended_steps": (
            "1. Send formal acknowledgement within 5 business days.\n"
            "2. Share DHL's latest Sustainability Report and GoGreen targets.\n"
            "3. Offer to schedule an information session with the Sustainability team.\n"
            "4. Log the concern in the stakeholder engagement register."
        ),
        "escalate_to_council": False,
        "timeline": "Within 5 business days"
    },
    ("Regulator", "Compliance"): {
        "engagement_format": "Consultation",
        "recommended_steps": (
            "1. Acknowledge receipt within 48 hours.\n"
            "2. Assign to Legal & Compliance team for review.\n"
            "3. Prepare standard documentation package.\n"
            "4. Schedule follow-up call with the regulatory body."
        ),
        "escalate_to_council": False,
        "timeline": "Within 48 hours"
    },
    ("Investor", "Strategic"): {
        "engagement_format": "Information & Communication",
        "recommended_steps": (
            "1. Forward to Investor Relations team immediately.\n"
            "2. Include concern in next quarterly stakeholder update.\n"
            "3. Prepare a written summary of DHL's position on the topic.\n"
            "4. Offer bilateral meeting at next investor roadshow."
        ),
        "escalate_to_council": False,
        "timeline": "Within 1 week"
    },
    ("Media", "Reputational"): {
        "engagement_format": "Information & Communication",
        "recommended_steps": (
            "1. Route to Corporate Communications team within 24 hours.\n"
            "2. Prepare holding statement using approved messaging guidelines.\n"
            "3. Do not engage directly until Communications team approves.\n"
            "4. Monitor media channels for publication."
        ),
        "escalate_to_council": False,
        "timeline": "Within 24 hours"
    },
    ("Supplier", "Operational"): {
        "engagement_format": "Consultation",
        "recommended_steps": (
            "1. Acknowledge and log issue in supplier management system.\n"
            "2. Schedule a consultation call with the relevant operations team.\n"
            "3. Review supplier contract terms if applicable.\n"
            "4. Issue a formal written response within 2 weeks."
        ),
        "escalate_to_council": False,
        "timeline": "Within 2 weeks"
    },
    # Fallback for any unmatched combination
    "*": {
        "engagement_format": "Consultation",
        "recommended_steps": (
            "1. Acknowledge receipt of the stakeholder message within 5 days.\n"
            "2. Route to the identified owning department for review.\n"
            "3. Prepare a standard written response.\n"
            "4. Log the issue in the stakeholder engagement register per DHL guidelines."
        ),
        "escalate_to_council": False,
        "timeline": "Within 5 business days"
    }
}

def lookup_playbook(stakeholder_group: str, issue_type: str) -> dict:
    """Look up playbook entry. Falls back to '*' if no exact match found."""
    key = (stakeholder_group, issue_type)
    return PLAYBOOK.get(key, PLAYBOOK["*"])


# HELPER — safe JSON parse with regex fallback

def safe_json_parse(text: str, fallback: dict) -> dict:
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return fallback


# 3. AGENT 1 — CLASSIFIER
# Identifies issue type, stakeholder group, owning department.

@traceable(name="Agent1_Classifier", run_type="chain",
           metadata={"agent_role": "classifier", "guideline_section": "§3"})

def classifier_agent(state: TriageState) -> dict:
    # print("\n" + "─"*65)
    # print("AGENT 1 — CLASSIFIER")
    # print("─"*65)

    docs = retriever.invoke("stakeholder groups owning department responsibility")
    doc_context = "\n\n".join(d.page_content for d in docs)

    system_prompt = f"""You are a classification specialist for DHL Group's
stakeholder engagement system. Use DHL's guideline as reference:
{doc_context}

Classify the incoming message. Return ONLY a JSON object with exactly these keys:
{{
  "issue_type": "one of: Environmental | Compliance | Reputational | Strategic | Operational | Legal",
  "stakeholder_group": "one of: Customer | Employee | Investor | NGO | Regulator | Supplier | Media | Academia | Business Association | Strategic Partner",
  "owning_department": "the DHL department that should own this based on the guideline"
}}
No explanation. No markdown. Only the JSON."""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Message:\n{state['stakeholder_message']}")
    ])

    result = safe_json_parse(response.content, {
        "issue_type": "Unknown",
        "stakeholder_group": "Unknown",
        "owning_department": "Corporate Sustainability Office"
    })

    # print(f"   Issue type      : {result['issue_type']}")
    # print(f"   Stakeholder     : {result['stakeholder_group']}")
    # print(f"   Owning dept     : {result['owning_department']}")

    return {
        "issue_type":        result["issue_type"],
        "stakeholder_group": result["stakeholder_group"],
        "owning_department": result["owning_department"],
        "steps_taken": [f"Agent 1 — Classifier: {result['stakeholder_group']} / {result['issue_type']} → {result['owning_department']}"]
    }


# 4. AGENT 2 — RISK ASSESSOR  (+ conditional edge)
# Evaluates urgency, risk type, ramifications.

@traceable(
    name="Agent2_RiskAssessor",
    run_type="chain",
    metadata={"agent_role": "risk_assessor", "guideline_section": "§6"}
)
def risk_assessor_agent(state: TriageState) -> dict:
    # print("\n" + "─"*65)
    # print("AGENT 2 — RISK ASSESSOR")
    # print("─"*65)

    docs = retriever.invoke("urgency ramifications strategic decision-making")
    doc_context = "\n\n".join(d.page_content for d in docs)

    system_prompt = f"""You are a risk assessment specialist for DHL Group.
Section 6 of DHL's guideline requires you to ascertain urgency and ramifications:
{doc_context}

Known classification:
- Issue type   : {state['issue_type']}
- Stakeholder  : {state['stakeholder_group']}
- Owning dept  : {state['owning_department']}

Assess the risk. Return ONLY a JSON object:
{{
  "urgency_level": "one of: Low | Medium | High | Critical",
  "risk_type": "one of: PR | Regulatory | Strategic | Operational | Legal | Reputational",
  "ramifications": "2-3 sentences on what could go wrong if DHL does not respond"
}}
No explanation. No markdown. Only the JSON."""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Message:\n{state['stakeholder_message']}")
    ])

    result = safe_json_parse(response.content, {
        "urgency_level": "Medium",
        "risk_type": "Reputational",
        "ramifications": "Could impact stakeholder trust if not addressed promptly."
    })

    urgency_emoji = {"Low": "🟢", "Medium": "🟡", "High": "🟠", "Critical": "🔴"}.get(result["urgency_level"], "⚪")
    # print(f"   Urgency level   : {urgency_emoji} {result['urgency_level']}")
    # print(f"   Risk type       : {result['risk_type']}")
    # print(f"   Ramifications   : {result['ramifications'][:80]}...")
    # print(f"\n Routing decision: {'LLM reasoning (Action Advisor)' if result['urgency_level'] in ['High', 'Critical'] else 'Playbook Matcher'}")

    return {
        "urgency_level": result["urgency_level"],
        "risk_type":     result["risk_type"],
        "ramifications": result["ramifications"],
        "steps_taken": [f"Agent 2 — Risk Assessor: {result['urgency_level']} urgency / {result['risk_type']} risk"]
    }


# 5a. AGENT 3a — ACTION ADVISOR ( in case of High / Critical path)
@traceable(
    name="Agent3a_ActionAdvisor",
    run_type="chain",
    metadata={"agent_role": "action_advisor", "path": "llm_reasoning"}
)
def action_advisor_agent(state: TriageState) -> dict:
    # print("\n" + "─"*65)
    # print("AGENT 3a — ACTION ADVISOR  [LLM reasoning path]")
    # print(f" Activated because urgency = {state['urgency_level']}")
    # print("─"*65)

    docs = retriever.invoke("engagement format consultation dialogue partnership course of action")
    doc_context = "\n\n".join(d.page_content for d in docs)

    prompt_version = state.get("action_advisor_prompt_version", "action_advisor_v2.0")
    prompt_config  = load_prompt(prompt_version)
    system_prompt  = prompt_config["template"].format(
        doc_context=doc_context,
        issue_type=state["issue_type"],
        stakeholder_group=state["stakeholder_group"],
        owning_department=state["owning_department"],
        urgency=state["urgency_level"],
        risk_type=state["risk_type"],
        ramifications=state["ramifications"]
    )
    print(f"    Prompt version  : {prompt_version}")

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Message:\n{state['stakeholder_message']}")
    ])

    result = safe_json_parse(response.content, {
        "engagement_format": "Formal Stakeholder Dialogue",
        "recommended_steps": "1. Escalate immediately\n2. Convene emergency response team\n3. Issue holding statement\n4. Schedule formal dialogue",
        "escalate_to_council": True,
        "timeline": "Within 24 hours"
    })

    # print(f"   Engagement fmt  : {result['engagement_format']}")
    # print(f"   Timeline        : {result['timeline']}")
    # print(f"   Escalate?       : {result['escalate_to_council']}")

    return {
        "engagement_format":   result["engagement_format"],
        "recommended_steps":   result["recommended_steps"],
        "escalate_to_council": result["escalate_to_council"],
        "timeline":            result["timeline"],
        "path_taken":          "llm_reasoning",
        "steps_taken": [f"Agent 3a — Action Advisor (LLM): '{result['engagement_format']}' / respond {result['timeline']}"]
    }


# 5b. AGENT 3b — PLAYBOOK MATCHER  ( in case of Low / Medium path)

@traceable(
    name="Agent3b_PlaybookMatcher",
    run_type="tool",                        # 'tool' not 'chain' — no LLM involved
    metadata={"agent_role": "playbook_matcher", "path": "deterministic"}
)
def playbook_matcher_agent(state: TriageState) -> dict:
    # print("\n" + "─"*65)
    # print(" AGENT 3b — PLAYBOOK MATCHER  [deterministic path]")
    # print(f" Activated because urgency = {state['urgency_level']}")
    # print("─"*65)

    result = lookup_playbook(state["stakeholder_group"], state["issue_type"])
    matched_key = (state["stakeholder_group"], state["issue_type"])
    used_fallback = matched_key not in PLAYBOOK

    # print(f"   Playbook key    : {matched_key} {'(fallback used)' if used_fallback else '(exact match)'}")
    # print(f"   Engagement fmt  : {result['engagement_format']}")
    # print(f"   Timeline        : {result['timeline']}")
    # print(f"   A deterministic response with no LLM call")

    return {
        "engagement_format":   result["engagement_format"],
        "recommended_steps":   result["recommended_steps"],
        "escalate_to_council": result["escalate_to_council"],
        "timeline":            result["timeline"],
        "path_taken":          "playbook",
        "steps_taken": [f"Agent 3b — Playbook Matcher: exact match={not used_fallback} / '{result['engagement_format']}' / {result['timeline']}"]
    }


# 6. REPORT COMPILER

def report_compiler_node(state: TriageState) -> dict:
    print("\n" + "─"*65)
    print(" REPORT COMPILER")
    print("─"*65)

    urgency_emoji = {"Low": "🟢", "Medium": "🟡", "High": "🟠", "Critical": "🔴"}.get(state["urgency_level"], "⚪")
    escalation_line = (
        "🔴 YES — escalate to Sustainability Advisory Council"
        if state["escalate_to_council"]
        else "🟢 NO — handle at department level"
    )
    path_line = (
        " LLM reasoning  (High/Critical urgency — novel situation)"
        if state["path_taken"] == "llm_reasoning"
        else " Playbook match (Low/Medium urgency — standard template)"
    )

    report = f"""
╔══════════════════════════════════════════════════════════════╗
║           DHL GROUP — STAKEHOLDER TRIAGE REPORT              ║
╚══════════════════════════════════════════════════════════════╝

📨  INCOMING MESSAGE
{state['stakeholder_message']}

──────────────────────────────────────────────────────────────
🔍  CLASSIFICATION  [Agent 1]
──────────────────────────────────────────────────────────────
  Issue Type        : {state['issue_type']}
  Stakeholder Group : {state['stakeholder_group']}
  Owning Department : {state['owning_department']}

──────────────────────────────────────────────────────────────
{urgency_emoji}  RISK ASSESSMENT  [Agent 2]
──────────────────────────────────────────────────────────────
  Urgency Level     : {state['urgency_level']}
  Risk Type         : {state['risk_type']}
  Ramifications     : {state['ramifications']}

──────────────────────────────────────────────────────────────
📋  COURSE OF ACTION  [Agent 3 — {path_line}]
──────────────────────────────────────────────────────────────
  Engagement Format : {state['engagement_format']}
  Response Timeline : {state['timeline']}
  Escalate to SAC   : {escalation_line}

  Recommended Steps:
{state['recommended_steps']}

──────────────────────────────────────────────────────────────
🔁  ORCHESTRATION AUDIT TRAIL
──────────────────────────────────────────────────────────────
{chr(10).join(f"  {i+1}. {s}" for i, s in enumerate(state['steps_taken']))}

  Guideline ref  : DHL Stakeholder Engagement Guideline (08/2025)
  Sections used  : §3 stakeholder groups · §5 engagement formats
                   §6 ownership & course of action
╚══════════════════════════════════════════════════════════════╝"""

    print(report)
    return {
        "triage_report": report,
        "steps_taken": ["Report Compiler: triage report generated"]
    }


# 7. CONDITIONAL EDGE FUNCTION
# This is the routing logic between Agent 2 and Agent 3.
# LangGraph calls this after risk_assessor_agent completes and uses
# the return value to decide which node to execute next.

def route_by_urgency(state: TriageState) -> str:
    """
    Conditional edge: reads urgency_level from shared state.
    High or Critical → LLM reasoning (Action Advisor)
    Low or Medium    → deterministic lookup (Playbook Matcher)
    """
    if state["urgency_level"] in ["High", "Critical"]:
        return "action_advisor"
    return "playbook_matcher"


# 8. BUILD THE LANGGRAPH

workflow = StateGraph(TriageState)

# Add all nodes
workflow.add_node("classifier",      classifier_agent)
workflow.add_node("risk_assessor",   risk_assessor_agent)
workflow.add_node("action_advisor",  action_advisor_agent)   # High/Critical
workflow.add_node("playbook_matcher",playbook_matcher_agent) # Low/Medium
workflow.add_node("report_compiler", report_compiler_node)

# Entry point
workflow.set_entry_point("classifier")

# Linear edge: classifier → risk_assessor
workflow.add_edge("classifier", "risk_assessor")

# *** CONDITIONAL EDGE ***
workflow.add_conditional_edges(
    "risk_assessor",        # source node
    route_by_urgency,       # function that returns the next node name
    {
        "action_advisor":   "action_advisor",   # High/Critical
        "playbook_matcher": "playbook_matcher"  # Low/Medium
    }
)

# Both paths converge at report_compiler
workflow.add_edge("action_advisor",  "report_compiler")
workflow.add_edge("playbook_matcher","report_compiler")
workflow.add_edge("report_compiler", END)

triage_app = workflow.compile()

print(" Stakeholder Triage System compiled")



 Stakeholder Triage System compiled


In [59]:
# 9. TEST SCENARIOS

test_messages = [
    {
        "label": "Scenario A — NGO / LLM path expected",
        "message": (
            "We are Greenwatch, an environmental NGO. We have been monitoring "
            "DHL's carbon emissions in Southeast Asia and plan to publish a "
            "critical report next month unless we receive a formal response. "
            "We are also in contact with three EU regulators about our findings."
        )
    },
    {
        "label": "Scenario B — Regulator / LLM path expected",
        "message": (
            "This is the European Commission Trade Compliance Unit. We are "
            "initiating formal enforcement proceedings against DHL Group for "
            "non-compliance with supplier due diligence requirements. You have "
            "30 days to respond or face financial penalties and operational restrictions."
        )
    },
    {
        "label": "Scenario C — Investor / Playbook path expected",
        "message": (
            "As a minority institutional investor, we would appreciate receiving "
            "DHL's latest sustainability report and any updates on your Strategy "
            "2030 progress when these become available at your convenience."
        )
    },
    {
        "label": "Scenario D — Supplier /  Playbook path expected",
        "message": (
            "We are a logistics subcontractor in Germany. We have some questions "
            "about the updated operational guidelines for last-mile delivery and "
            "would like to discuss these with the relevant team whenever it is possible."
        )
    }
]

for scenario in test_messages:
    print(f"  {scenario['label']}")

    initial_state = TriageState(
        stakeholder_message=scenario["message"],
        issue_type="", stakeholder_group="", owning_department="",
        urgency_level="", risk_type="", ramifications="",
        engagement_format="", recommended_steps="",
        escalate_to_council=False, timeline="",
        path_taken="", triage_report="",
        steps_taken=[]
    )

    result = triage_app.invoke(initial_state)




  Scenario A — NGO / LLM path expected
    Prompt version  : action_advisor_v2.0

─────────────────────────────────────────────────────────────────
 REPORT COMPILER
─────────────────────────────────────────────────────────────────

╔══════════════════════════════════════════════════════════════╗
║           DHL GROUP — STAKEHOLDER TRIAGE REPORT              ║
╚══════════════════════════════════════════════════════════════╝

📨  INCOMING MESSAGE
We are Greenwatch, an environmental NGO. We have been monitoring DHL's carbon emissions in Southeast Asia and plan to publish a critical report next month unless we receive a formal response. We are also in contact with three EU regulators about our findings.

──────────────────────────────────────────────────────────────
🔍  CLASSIFICATION  [Agent 1]
──────────────────────────────────────────────────────────────
  Issue Type        : Environmental
  Stakeholder Group : NGO
  Owning Department : Corporate Sustainability Office

───────────────────

##  RAGAS evaluation for SYSTEM 1 (prompt v1 vs v2)

Answer Relevance, Faithfulness, and Context Precision for both prompt versions.

In [39]:
from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset
import json

# wrap BOTH the LLM and embeddings for RAGAS
# RAGAS needs its own LLM wrapper (separate from the chain's LLM)
# AND its own embeddings wrapper for the answer_relevancy metric
# Without the embeddings wrapper, it silently falls back to OpenAI

ragas_llm = LangchainLLMWrapper(
    ChatGroq(
        model_name="openai/gpt-oss-120b",
        temperature=0,
        groq_api_key=GROQ_API_KEY
    )
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
)

print("RAGAS LLM wrapper: Groq llama-3.1-8b-instant")
print("RAGAS embeddings wrapper: BGE-small-en-v1.5")

#  Test set
test_set = [
    {
        "question": "What are the three guiding principles for stakeholder engagement at DHL?",
        "ground_truth": "The three guiding principles are Inclusivity, Materiality, and Responsiveness, based on the AA1000APS Accountability Principles."
    },
    {
        "question": "What are the four stages of the AA1000SES stakeholder engagement process?",
        "ground_truth": "The four stages are: 1. Planning, 2. Preparing, 3. Implementing the engagement format, 4. Reviewing and evaluating."
    },
    {
        "question": "Who is responsible for managing stakeholder engagement at DHL?",
        "ground_truth": "The department accountable for the respective stakeholder group, with support from the Corporate Sustainability Office."
    },
    {
        "question": "What is the purpose of DHL's Sustainability Advisory Council?",
        "ground_truth": "To better understand and consider global trends and stakeholders' views on sustainability."
    },
    {
        "question": "What are DHL's four bottom lines according to Strategy 2030?",
        "ground_truth": "Provider of Choice, Employer of Choice, Investment of Choice, and Green Logistics of Choice."
    }
]

#  Build dataset for a given prompt version
def build_eval_dataset(prompt_version: str) -> Dataset:
    chain = build_rag_chain(prompt_version)
    questions, answers, contexts, ground_truths = [], [], [], []

    for item in test_set:
        q = item["question"]
        retrieved_docs = retriever.invoke(q)
        ctx = [doc.page_content for doc in retrieved_docs]
        ans = chain.invoke(q)

        questions.append(q)
        answers.append(ans)
        contexts.append(ctx)
        ground_truths.append(item["ground_truth"])

    return Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    })

#  Run evaluation for both prompt versions
metrics = [answer_relevancy, faithfulness, context_precision]

# Pass ragas_llm and ragas_embeddings explicitly to every metric
for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ragas_embeddings

print("\n Evaluating prompt v1...")
dataset_v1 = build_eval_dataset("rag_v1.0")
results_v1 = evaluate(dataset_v1, metrics=metrics,  llm=ragas_llm,
    embeddings=ragas_embeddings)

print(" Evaluating prompt v2...")
dataset_v2 = build_eval_dataset("rag_v2.0")
results_v2 = evaluate(dataset_v2, metrics=metrics,  llm=ragas_llm,
    embeddings=ragas_embeddings)



/tmp/ipykernel_2134/234910766.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_relevancy, faithfulness, context_precision
/tmp/ipykernel_2134/234910766.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import answer_relevancy, faithfulness, context_precision
/tmp/ipykernel_2134/234910766.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import answer_relevancy, faithfulness, context_preci

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_2134/234910766.py:23: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


RAGAS LLM wrapper: Groq llama-3.1-8b-instant
RAGAS embeddings wrapper: BGE-small-en-v1.5

 Evaluating prompt v1...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

 Evaluating prompt v2...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[9]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[6]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})


In [60]:
#  Print comparison table
metric_keys = ["answer_relevancy", "faithfulness", "context_precision"]
print("\n" + "="*55)
print("RAGAS RESULTS — Prompt v1 vs v2")
print("="*55)
print(f"{'Metric':<25} {'v1':>8} {'v2':>8} {'Change':>10}")
print("-"*55)
for key in metric_keys:
    v1 = float(results_v1.to_pandas()[key].mean())
    v2 = float(results_v2.to_pandas()[key].mean())
    delta = v2 - v1
    arrow = "↑" if delta > 0 else "↓"
    print(f"{key:<25} {v1:>8.3f} {v2:>8.3f} {arrow}{abs(delta):>+8.3f}")

# Save to JSON
eval_results = {
    "rag_v1": {k: results_v1[k] for k in metric_keys},
    "rag_v2": {k: results_v2[k] for k in metric_keys}
}
with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print("\n Done")
print(" Saved to eval_results.json")


RAGAS RESULTS — Prompt v1 vs v2
Metric                          v1       v2     Change
-------------------------------------------------------
answer_relevancy             0.957    0.940 ↓  +0.017
faithfulness                 0.760    0.843 ↑  +0.083
context_precision            1.000    0.800 ↓  +0.200

 Done
 Saved to eval_results.json


Now I update the EVALUTION SCORES in the prompt  registery  




In [61]:
update_eval_scores("rag_v1.0", {k: float(results_v1.to_pandas()[k].mean()) for k in metric_keys})
update_eval_scores("rag_v2.0", {k: float(results_v2.to_pandas()[k].mean()) for k in metric_keys})

Scores linked to rag_v1.0: {'answer_relevancy': 0.9567654428766209, 'faithfulness': 0.76, 'context_precision': 0.9999999999133333}
Scores linked to rag_v2.0: {'answer_relevancy': 0.9396014367607646, 'faithfulness': 0.8433333333333334, 'context_precision': 0.7999999999333334}


Evaluation using LLM-as-Judge for SYSTEM TWO

**evaluate only the Action Advisor(Agent three)**

In [63]:
# Compares action_advisor_v1.0 vs action_advisor_v2.0

from langsmith import traceable


eval_messages = [

                 {
        "label": "Academia Strategic Low urgency",
        "stakeholder_message": (
            "We are a research team at the University of Bonn studying sustainable "
            "logistics practices in Germany. We would appreciate the opportunity to "
            "collaborate with DHL on a study examining last-mile delivery emissions. "
            "We are happy to work around your team's availability."
        ),
        "issue_type":        "Strategic",
        "stakeholder_group": "Academia",
        "owning_department": "Corporate Sustainability Office",
        "urgency_level":     "Low",
        "risk_type":         "Strategic",
        "ramifications":     "A missed collaboration opportunity with a local university "
                             "relevant to DHL's GoGreen strategy and Strategy 2030 goals."
    },
                 {
        "label": "Media Reputational Medium urgency",
        "stakeholder_message": (
            "I am a journalist at the Financial Times researching a piece on working "
            "conditions in last-mile logistics across Europe. I would like to request "
            "a comment from DHL on reports of delivery drivers being classified as "
            "independent contractors without adequate benefits. Deadline is 2 weeks."
        ),
        "issue_type":        "Reputational",
        "stakeholder_group": "Media",
        "owning_department": "Corporate Communications",
        "urgency_level":     "Medium",
        "risk_type":         "PR",
        "ramifications":     "An FT article without DHL's response could frame the "
                             "story negatively and attract further regulatory scrutiny."
    }

]


docs = retriever.invoke(
    "engagement format consultation dialogue partnership course of action section 5 6"
)
doc_context = "\n\n".join(d.page_content for d in docs)


# ── LLM-as-Judge function ─────────────────────────────────────────────────────
@traceable(
    name="LLM_Judge_ActionAdvisor",
    run_type="llm",
    metadata={"role": "evaluator", "reference": "DHL Stakeholder Engagement Guideline"}
)
def llm_judge(
    stakeholder_message: str,
    urgency_level: str,
    action_advisor_output: dict,
    doc_context: str
) -> dict:
    """
    LLM-as-Judge: scores the Action Advisor output on 3 dimensions.
    No ground truth needed — the DHL document is the reference.

    Scoring dimensions:
      1. Guideline Faithfulness  — is the output grounded in the document?
      2. Urgency Appropriateness — is the response proportional to urgency?
      3. Action Concreteness     — are the steps specific and actionable?
    """
    judge_prompt = f"""You are an expert evaluator assessing the quality of
stakeholder engagement recommendations made by an AI system for DHL Group.

You have access to the official DHL Stakeholder Engagement Guideline:
{doc_context}

You are evaluating this AI-generated recommendation:
  Engagement format  : {action_advisor_output.get('engagement_format')}
  Recommended steps  : {action_advisor_output.get('recommended_steps')}
  Escalate to council: {action_advisor_output.get('escalate_to_council')}
  Timeline           : {action_advisor_output.get('timeline')}

This recommendation was generated for the following situation:
  Stakeholder message: {stakeholder_message}
  Urgency level      : {urgency_level}

Score the recommendation on these 3 dimensions from 1 to 5:

1. GUIDELINE FAITHFULNESS (1-5)
   Is every recommended step traceable to something in the DHL guideline?
   5 = every step cites or clearly reflects a specific section
   3 = some steps grounded, others are generic
   1 = recommendation ignores the guideline entirely

2. URGENCY APPROPRIATENESS (1-5)
   Is the timeline and engagement format proportional to the urgency level?
   5 = perfectly calibrated to the urgency
   3 = partially appropriate but timeline or format is off
   1 = completely disproportionate to the urgency

3. ACTION CONCRETENESS (1-5)
   Are the recommended steps specific and actionable?
   5 = each step is specific, assigned, and time-bound
   3 = steps are reasonable but vague
   1 = steps are too generic to act on

Return ONLY a JSON object:
{{
  "guideline_faithfulness": <1-5>,
  "urgency_appropriateness": <1-5>,
  "action_concreteness": <1-5>,
  "overall_score": <average of the three, rounded to 1 decimal>,
  "judge_reasoning": "2-3 sentences explaining the scores"
}}
No explanation. No markdown. Only the JSON."""

    response = llm.invoke([
        SystemMessage(content=judge_prompt),
        HumanMessage(content="Please evaluate the recommendation above.")
    ])

    return safe_json_parse(response.content, {
        "guideline_faithfulness":  3,
        "urgency_appropriateness": 3,
        "action_concreteness":     3,
        "overall_score":           3.0,
        "judge_reasoning":         "Could not parse judge response."
    })


# ── Run evaluation for both prompt versions ───────────────────────────────────
print("="*65)
print("  LLM-AS-JUDGE EVALUATION — Action Advisor v1.0 vs v2.0")
print("="*65)

# Collect scores across all messages for each version
all_scores = {"action_advisor_v1.0": [], "action_advisor_v2.0": []}

for msg in eval_messages:
    print(f"\n{'─'*65}")
    print(f"📨 {msg['label']}")
    print(f"{'─'*65}")

    for version in ["action_advisor_v1.0", "action_advisor_v2.0"]:

        # Step 1: Run Action Advisor with this prompt version

        initial_state = TriageState(
            stakeholder_message=msg["stakeholder_message"],
            action_advisor_prompt_version=version,  # ← controls which prompt runs
            issue_type=msg["issue_type"],
            stakeholder_group=msg["stakeholder_group"],
            owning_department=msg["owning_department"],
            urgency_level=msg["urgency_level"],
            risk_type=msg["risk_type"],
            ramifications=msg["ramifications"],
            # rest of state empty — only Action Advisor runs in eval
            engagement_format="", recommended_steps="",
            escalate_to_council=False, timeline="",
            path_taken="", triage_report="",
            steps_taken=[]
        )

        result = triage_app.invoke(initial_state)

        # Step 2: LLM judges the output
        scores = llm_judge(
            stakeholder_message=msg["stakeholder_message"],
            urgency_level=msg["urgency_level"],
            action_advisor_output=result,
            doc_context=doc_context
        )

        all_scores[version].append(scores)

        print(f"\n  [{version}]")
        print(f"  Engagement format : {result.get('engagement_format')}")
        print(f"  Timeline          : {result.get('timeline')}")
        print(f"  ── Judge scores ──")
        print(f"  Guideline faithful: {scores['guideline_faithfulness']}/5")
        print(f"  Urgency approp.   : {scores['urgency_appropriateness']}/5")
        print(f"  Action concrete   : {scores['action_concreteness']}/5")
        print(f"  Overall           : {scores['overall_score']}/5")
        print(f"  Reasoning         : {scores['judge_reasoning']}")


# ── Aggregate results ─────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  AGGREGATED RESULTS")
print(f"{'='*65}")
print(f"  {'Metric':<28} {'v1.0':>8} {'v2.0':>8} {'Winner':>10}")
print(f"  {'─'*28} {'─'*8} {'─'*8} {'─'*10}")

dimensions = ["guideline_faithfulness", "urgency_appropriateness",
              "action_concreteness", "overall_score"]

agg = {}
for version in ["action_advisor_v1.0", "action_advisor_v2.0"]:
    agg[version] = {
        dim: round(
            sum(s[dim] for s in all_scores[version]) / len(all_scores[version]), 2
        )
        for dim in dimensions
    }

for dim in dimensions:
    v1 = agg["action_advisor_v1.0"][dim]
    v2 = agg["action_advisor_v2.0"][dim]
    winner = "v2.0 ✅" if v2 > v1 else ("v1.0 ✅" if v1 > v2 else "tie")
    print(f"  {dim:<28} {v1:>8.2f} {v2:>8.2f} {winner:>10}")

# ── Link scores back to prompt registry ──────────────────────────────────────
update_eval_scores("action_advisor_v1.0", {
    d: agg["action_advisor_v1.0"][d] for d in dimensions
})
update_eval_scores("action_advisor_v2.0", {
    d: agg["action_advisor_v2.0"][d] for d in dimensions
})

print(f"\n✅ Scores linked back to prompt registry")
#print(f"   Call print_registry_summary() to see the full version history")

  LLM-AS-JUDGE EVALUATION — Action Advisor v1.0 vs v2.0

─────────────────────────────────────────────────────────────────
📨 Academia Strategic Low urgency
─────────────────────────────────────────────────────────────────

─────────────────────────────────────────────────────────────────
 REPORT COMPILER
─────────────────────────────────────────────────────────────────

╔══════════════════════════════════════════════════════════════╗
║           DHL GROUP — STAKEHOLDER TRIAGE REPORT              ║
╚══════════════════════════════════════════════════════════════╝

📨  INCOMING MESSAGE
We are a research team at the University of Bonn studying sustainable logistics practices in Germany. We would appreciate the opportunity to collaborate with DHL on a study examining last-mile delivery emissions. We are happy to work around your team's availability.

──────────────────────────────────────────────────────────────
🔍  CLASSIFICATION  [Agent 1]
───────────────────────────────────────────────────